# Forcing and Test Case Overview

3x12 panel plot:
- rows: journey_time_start
- columns: currents, winds, waves

Lines per panel:
- journey name (fwd / bwd)
- speeds
- ablation scenario

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

import pandas as pd
import geopandas as gpd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
import cartopy
import cmocean
import xarray as xr

from load_tuning_results import (
    load_results,
    get_forcing_df,
    get_seed_routes_gdf,
    filter_suspicious_routes,
    add_derived_features,
)

import warnings

warnings.filterwarnings("ignore")

In [ ]:
# parameters

# extent
lon_min, lon_max = -80 - 5, -10 + 5
lat_min, lat_max = 25 - 5, 55 + 5

# results dataframe from
gpq_file = "../results/results_prelim.geoparquet"

# any single results file
results_file = "../results/results_ablation_baseline_20251218_103839.msgpack"

In [ ]:
gdf = gpd.read_parquet(gpq_file)
gdf = add_derived_features(gdf)
gdf = filter_suspicious_routes(gdf)
gdf

In [ ]:
# Load results and extract seed routes
results = load_results([results_file])
gdf_seed = get_seed_routes_gdf(results)

In [ ]:
# Extract forcing paths
forcing_df = get_forcing_df(results)
forcing_currents_path = forcing_df["forcing_currents_path"].iloc[0]
print(forcing_currents_path)
forcing_waves_path = forcing_df["forcing_waves_path"].iloc[0]
print(forcing_waves_path)
forcing_winds_path = forcing_df["forcing_winds_path"].iloc[0]
print(forcing_winds_path)

In [ ]:
from ship_routing.core.data import load_currents, load_waves, load_winds

In [ ]:
ds_currents = load_currents(Path("..") / forcing_currents_path)
ds_waves = load_waves(Path("..") / forcing_waves_path)
ds_winds = load_winds(Path("..") / forcing_winds_path)

In [ ]:
ds_currents = ds_currents.assign(
    speed=(ds_currents.to_array() ** 2).sum("variable") ** 0.5
)
ds_currents

In [ ]:
ds_winds = ds_winds.resample(time="1D").mean()
ds_winds

In [ ]:
ds_winds = ds_winds.assign(speed=(ds_winds.to_array() ** 2).sum("variable") ** 0.5)
ds_winds

In [ ]:
times = sorted(list(gdf.journey_time_start.unique()))
times

In [ ]:
monthly_mean_current_speed = ds_currents.speed.resample(time="1MS").mean()
monthly_q90_wave_height = ds_waves.wh.resample(time="1MS").quantile(0.9)
monthly_q90_wind_speed = ds_winds.speed.resample(time="1MS").quantile(0.9)

In [ ]:
lon_cent, lat_cent = (
    gdf_seed.iloc[0].geometry.centroid.x,
    gdf_seed.iloc[0].geometry.centroid.y,
)
lon_cent, lat_cent

In [ ]:
monthly_mean_current_speed_ranges = (
    0,
    monthly_mean_current_speed.quantile(0.98).data[()],
)
monthly_q90_wave_height_ranges = (0, monthly_q90_wave_height.quantile(0.98).data[()])
monthly_q90_wind_speed_ranges = (0, monthly_q90_wind_speed.quantile(0.98).data[()])

In [ ]:
gdf.forcing_scenario_name.unique()

In [ ]:
# journey_speed_knots_linewidth = {8.0: .5, 10.0: .5, 12.0: .5}
journey_speed_knots_linewidth = {12.0: 1.7}
journey_name_linestyle = {"Atlantic_backward": "--", "Atlantic_forward": "-"}
forcing_scenario_name_color = {
    "baseline": "magenta",
    "no_currents": "cyan",
    "no_waves": "darkgray",
    "no_winds": "orange",
}

In [ ]:
_gdf = (
    gdf.sort_values(by="elite_cost_absolute")
    .groupby(
        [
            "journey_speed_knots",
            "journey_name",
            "forcing_scenario_name",
            "journey_time_start",
        ]
    )
    .first()
)

In [ ]:
lon_min, lat_min = _gdf.bounds.filter(like="min").min()
lon_max, lat_max = _gdf.bounds.filter(like="max").max()

In [ ]:
fig, ax = plt.subplots(
    len(times),
    3,
    subplot_kw={
        "projection": cartopy.crs.Stereographic(
            central_latitude=lat_cent, central_longitude=lon_cent
        )
    },
    figsize=(3 * 4, len(times) * 2.5),
    sharex=True,
    sharey=True,
)
fig.set_dpi(300)

for n in tqdm(range(len(times))):
    _time = times[n]
    monthly_mean_current_speed.isel(time=n).plot(
        vmin=monthly_mean_current_speed_ranges[0],
        vmax=monthly_mean_current_speed_ranges[1],
        extend="max",
        cmap=cmocean.cm.speed,
        ax=ax[n, 0],
        transform=cartopy.crs.PlateCarree(),
        rasterized=True,
        add_colorbar=False,
    )
    ax[n, 0].set_title(
        f"current speed | {_time[:7]} | max {monthly_mean_current_speed_ranges[1]:.1f} m/s"
    )

    monthly_q90_wave_height.isel(time=n).plot(
        vmin=monthly_q90_wave_height_ranges[0],
        vmax=monthly_q90_wave_height_ranges[1],
        extend="max",
        cmap=cmocean.cm.amp,
        ax=ax[n, 1],
        transform=cartopy.crs.PlateCarree(),
        rasterized=True,
        add_colorbar=False,
    )
    ax[n, 1].set_title(
        f"wave height q90 | {_time[:7]} | max {monthly_q90_wave_height_ranges[1]:.1f} m"
    )

    monthly_q90_wind_speed.isel(time=n).plot(
        vmin=monthly_q90_wind_speed_ranges[0],
        vmax=monthly_q90_wind_speed_ranges[1],
        extend="max",
        cmap=cmocean.cm.speed,
        ax=ax[n, 2],
        transform=cartopy.crs.PlateCarree(),
        rasterized=True,
        add_colorbar=False,
    )
    ax[n, 2].set_title(
        f"wind speed q90 | {_time[:7]} | max {monthly_q90_wind_speed_ranges[1]:.1f} m/s"
    )

    for m in range(3):
        for jsk, _linewidth in journey_speed_knots_linewidth.items():
            for jn, _linestyle in journey_name_linestyle.items():
                for fsn, _color in forcing_scenario_name_color.items():
                    try:
                        ax[n, m].plot(
                            *_gdf.loc[jsk].loc[jn].loc[fsn].loc[_time].geometry.xy,
                            transform=cartopy.crs.PlateCarree(),
                            color=_color,
                            linewidth=_linewidth,
                            linestyle=_linestyle,
                        )
                    except:
                        pass
        ax[n, m].coastlines()
        ax[n, m].gridlines(draw_labels=False)
        ax[n, m].set_extent([lon_min, lon_max - 1, lat_min - 5, lat_max])


style_handles = [
    Line2D(
        [0],
        [0],
        color="black",
        linestyle=_linestyle,
        linewidth=1.7,
        label=jn.replace("_", " "),
    )
    for jn, _linestyle in journey_name_linestyle.items()
]
color_handles = [
    Line2D(
        [0],
        [0],
        color=_color,
        linestyle="-",
        linewidth=1.7,
        label=fsn.replace("_", " "),
    )
    for fsn, _color in forcing_scenario_name_color.items()
]
legend_handles = style_handles + color_handles

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=len(legend_handles),
    frameon=False,
    bbox_to_anchor=(0.5, 0.02),
)
fig.tight_layout(rect=[0, 0.03, 1, 1])

fig.savefig("../figures/020_forcing_data_and_scenario_overview_routes.png", dpi=150)
fig.savefig("../figures/020_forcing_data_and_scenario_overview_routes.pdf", dpi=150)

In [ ]:
# Save best elites per test case for downstream analysis
gdf_best = _gdf.reset_index()
gdf_best.to_parquet("../results/best_elites_per_test_case.geoparquet")